# Classify two-dimensional observations

Fit a two-qubit classifier on six labeled observations, then inspect scores and a confusion matrix on held-out inputs. This tiny dataset lets you inspect every stage of supervised training.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)
figures = Path("figures")
figures.mkdir(exist_ok=True)

## Configure the experiment

The observations in a minibatch share parameters. Here NeuralEmbedding projects two features onto each wire. Reuploading allocates another projection at the second layer. The initializer dictionaries control both families independently.

In [ ]:
from qasp import Normal, Uniform
from qasp import create_classifier, ClassifierTrainer, draw_classifier

torch.manual_seed(2026)
# Small synthetic API fixture, not an accuracy benchmark.
X_train = np.array([[-0.8, -0.5], [-0.3, -0.7], [-0.5, 0.1],
                    [0.4, 0.8], [0.9, 0.2], [0.3, 0.5]], dtype=np.float32)
y_train = np.array([0, 0, 0, 1, 1, 1])
X_test = np.array([[-0.6, -0.2], [0.6, 0.4]], dtype=np.float32)
y_test = np.array([0, 1])

config = {
    "feature_map": "NeuralEmbedding",
    "feature_map_config": {"features": 2, "rotations": "Y"},
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 2, "rotations": "YZ", "reuploading": True,
        "include_imprimitive": True, "imprimitive": "CZ",
        "imprimitive_pattern": "chain",
    },
    "readout": "expectation", "observables": ["Z 0", "ZZ 0 1"],
    "init": Normal(0, 0.1), "input_scaling_init": Uniform(0.8, 1.2),
    "device": "default.qubit",
    "diff_method": "backprop", "shots": None,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
model = create_classifier(config, n_qubits=2, n_classes=2)
print(type(model).__name__)
print("Trainable scalars:", sum(p.numel() for p in model.parameters()))
print("Initial angles:", model.qlayer.params.detach())
print("Initial encoding weights:", model.qlayer.input_scaling_params.detach())

## Train

Hinge loss trains the expectation score as a margin. The trainer handles minibatches, differentiation, and validation. The stored history gives one training loss per epoch.

In [ ]:
trainer = ClassifierTrainer(model, objective="hinge", learning_rate=0.03)
history = trainer.fit(
    X_train, y_train, epochs=12, batch_size=3, shuffle=False,
    validation_data=(X_test, y_test), verbose=0,
)
print("Training losses:", history["train_loss"])

## Read the results

For a binary classifier the columns of predict_proba correspond to labels 0 and 1. Compare the predictions against the confusion matrix; two test points are enough to check wiring but too few to estimate generalization.

In [ ]:
report = trainer.evaluate(X_test, y_test)
print("Test loss:", report["loss"], "accuracy:", report["accuracy"])
print("Confusion matrix:", report["confusion_matrix"])
with torch.no_grad():
    probabilities = model.predict_proba(torch.as_tensor(X_test))
    predictions = model.predict(torch.as_tensor(X_test))
print(probabilities, predictions)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, len(history["train_loss"])+1), history["train_loss"], marker="o")
ax.set(xlabel="Epoch", ylabel="Training hinge loss")
plt.show()
assert torch.allclose(probabilities.sum(-1), torch.ones_like(probabilities[:, 0]))

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_classifier(model, inputs=X_test[0])
display(fig)
plt.close(fig)

### Export a paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

In [ ]:
import shutil
import subprocess

def show_tikz(source, name):
    """Write standalone Quantikz and, when available, render its PDF inline."""
    tex = figures / f"{name}.tex"
    tex.write_text(source, encoding="utf-8")
    print("LaTeX:", tex)
    if shutil.which("pdflatex") is None:
        print("Install a TeX distribution with Quantikz to compile the PDF.")
        return
    subprocess.run(["pdflatex", "-interaction=batchmode", "-halt-on-error", tex.name],
                   cwd=figures, check=True)
    pdf = tex.with_suffix(".pdf")
    print("Vector PDF:", pdf)
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if gs:
        png = subprocess.check_output([gs, "-q", "-sDEVICE=png16m", "-r120",
                                      "-o", "-", pdf.name], cwd=figures)
        display(Image(data=png))

In [ ]:
tex = draw_classifier(
    model, inputs=X_test[0], output="tikz",
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)
show_tikz(tex, "classifier")

## Try it

Replace Uniform(0.8, 1.2) with Ones() and reconstruct the classifier. Compare the initial probabilities, then the training curves.